# Make Jarvis's release signing key (no laptop needed)

Run the cells top to bottom (▶ on each). At the end two files download to your Chromebook:
- `jarvis-release.jks` — **the key. Back it up in two places** (Google Drive/USB stick + a password manager for the password). Lose it and you can never update the app on that phone.
- `key.txt` — the same key as text, to paste into GitHub. Delete it after.

Privacy: the key is made on Google's Colab computer, which is wiped when you close this tab. If that bothers you, make it on any computer with Java instead (see RELEASE_SIGNING.md).

In [ ]:
# 1. Java's keytool (Colab usually needs it installed; takes ~1 min)
import shutil, subprocess
if not shutil.which('keytool'):
    subprocess.run('apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null', shell=True, check=True)
print('keytool ready:', shutil.which('keytool'))

In [ ]:
# 2. Pick a password (at least 6 characters). Type it twice. WRITE IT DOWN.
from getpass import getpass
pw = getpass('Password: ')
assert len(pw) >= 6, 'Too short: at least 6 characters. Run this cell again.'
assert pw == getpass('Same password again: '), 'They did not match. Run this cell again.'
print('OK')

In [ ]:
# 3. Make the key
import os
if os.path.exists('jarvis-release.jks'): os.remove('jarvis-release.jks')
r = subprocess.run(['keytool', '-genkeypair', '-keystore', 'jarvis-release.jks', '-alias', 'jarvis',
                    '-keyalg', 'RSA', '-keysize', '4096', '-validity', '10000',
                    '-storepass', pw, '-keypass', pw, '-dname', 'CN=Jarvis'],
                   capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
assert r.returncode == 0 and os.path.getsize('jarvis-release.jks') > 1000, 'keytool failed (see above)'
print('Key made.')

In [ ]:
# 4. Text version for GitHub, then download both files
import base64
open('key.txt', 'w').write(base64.b64encode(open('jarvis-release.jks', 'rb').read()).decode())
print('key.txt is', os.path.getsize('key.txt'), 'characters')
from google.colab import files
files.download('jarvis-release.jks')
files.download('key.txt')

## Next (START_HERE.md, Part 7)
GitHub → the Jarvis repo → **Settings → Secrets and variables → Actions → New repository secret**, four times:

| Name | Value |
|---|---|
| `KEYSTORE_BASE64` | open `key.txt`, select all, copy, paste |
| `KEYSTORE_PASSWORD` | your password |
| `KEY_ALIAS` | `jarvis` |
| `KEY_PASSWORD` | your password (same one) |

Then delete `key.txt` from Downloads (keep `jarvis-release.jks` backed up).